In [15]:
import sys
from pathlib import Path

parent_path = Path().resolve().parent
sys.path.append(str(parent_path))
import util
import plotting.plotting_style as plt
import pandas as pd
import numpy as np
import seaborn as sns
from plotting.plotting_style import rgb
from scipy import stats
from matplotlib import ticker as mtick


# Manual path, automatic did not work
# data = util.load_data(path_file='C:\\Users\\Mattis\\OneDrive\\Kogni\\DataLiteracyProject\\Data-Literacy\\data_csv\\final_data_iaaf_scores_neu_v2.csv')
df = util.load_data() 

In [16]:
n_ = 35

df = df[df['jahr'] >= 2015].copy()

df['altersklasse'] = df['altersklasse'].replace({'Maenner': 'Adult', 'Frauen': 'Adult', '18': 'U18', '20': 'U20'})

# Get Data for Analysis, top 35, exclude U23
df = df.groupby(['jahr', 'altersklasse', 'geschlecht', 'disziplin']).filter(lambda x: x.name[1] != 'U23' and len(x) >= 35)

df = df.sort_values(by='iaaf_score', ascending=False)
df = df.groupby(['jahr', 'altersklasse', 'geschlecht', 'disziplin']).head(35)

# 1. Identifiziere alle Jahre im Datensatz
years = df['jahr'].unique()
num_years = len(years)

# 2. Finde Gruppen, die in JEDEM Jahr vorkommen
# Wir gruppieren und zählen, in wie vielen Jahren jede Gruppe auftaucht
group_counts = df.groupby(['altersklasse', 'geschlecht', 'disziplin'])['jahr'].nunique()

# 3. Filtere nur die Gruppen heraus, deren Anzahl an Jahren exakt der Gesamtzahl entspricht
stable_groups = group_counts[group_counts == num_years].index

# 4. Erstelle den bereinigten DataFrame
df_stable = df.set_index(['altersklasse', 'geschlecht', 'disziplin']).loc[stable_groups].reset_index()

# Kontrolle der Ergebnisse
print(f"Ursprüngliche Gruppen (inkl. Ausreißer): {len(group_counts)}")
print(f"Stabile Gruppen (in allen Jahren vorhanden): {len(stable_groups)}")
print(f"Entfernte Gruppen: {len(group_counts) - len(stable_groups)}")


category_stats = df_stable.groupby(['jahr', 'group'])['iaaf_score'].median().reset_index(name='iaaf_score_mean')
category_stats.rename(columns={'group': 'category'}, inplace=True)

total_stats = df_stable.groupby(['jahr'])['iaaf_score'].median().reset_index(name='iaaf_score_mean')
total_stats['category'] = 'GESAMT'

final_stats = pd.concat([category_stats, total_stats], ignore_index=True)

# 4. Relative Änderung zu 2015 berechnen
def calculate_pct_change(group):
    # Wir suchen den Wert von 2015 für diese spezifische Gruppe
    base_val = group.loc[group['jahr'] == 2015, 'iaaf_score_mean']
    if not base_val.empty:
        b = base_val.values[0]
        group['rel_change_pct'] = ((group['iaaf_score_mean'] - b) / b) * 100
    else:
        group['rel_change_pct'] = np.nan # Falls 2015 nicht existiert
    return group

# Sicherstellen, dass nach 'category' gruppiert wird
final_stats = final_stats.groupby('category', group_keys=False).apply(calculate_pct_change)



Ursprüngliche Gruppen (inkl. Ausreißer): 109
Stabile Gruppen (in allen Jahren vorhanden): 100
Entfernte Gruppen: 9


C:\Users\Mattis\AppData\Local\Temp\ipykernel_19008\3077564020.py:53: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_stats = final_stats.groupby('category', group_keys=False).apply(calculate_pct_change)


In [17]:


# --- 4. PLOTTING ---
with plt.rc_context(plt.bundles.icml2024(column='half')):
    fig, ax = plt.subplots()

    for cat in sorted(final_stats['category'].unique()):
        cat_data = final_stats[final_stats['category'] == cat].sort_values('jahr')
        
        
        ax.plot(cat_data['jahr'], cat_data['rel_change_pct'], 
                label=cat,
                marker='.')

    # Styling
    ax.axhline(0, color=plt.rgb.tue_dark, alpha=0.3, zorder=1)
    ax.axvline(2020, color='red', alpha=0.5, linestyle='--', zorder=1)

    ax.xaxis.set_major_locator(mtick.MultipleLocator(1))
    
    ax.set_ylabel('Relative Perfomance Change (\\%)')
    ax.grid(True, linestyle=':', alpha=0.3)

    # Legende
    ax.legend(title='Discipline category', ncols=2)

    plt.show()

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 975x602.583 with 1 Axes>

In [18]:
import matplotlib.ticker as mtick

# --- VORBEREITUNG (Farben & Styles) ---
colors = {
    'GESAMT': '#344953',
    'Sprint': '#e67e22',
    'Throw':  '#1abc9c',
    'Run':    '#3498db',
    'Jump':   '#e74c3c'
}

# Mapping für englische Labels
label_map = {
    'GESAMT': 'Overall',
    'Sprint': 'Sprint',
    'Throw':  'Throw',
    'Run':    'Run',
    'Jump':   'Jump'
}

column = "full"
baseline = False
# --- PLOTTING ---
with plt.rc_context(plt.bundles.icml2024(column=column)):
    fig, ax = plt.subplots()



    if baseline:
        ax.axvline(2015, color=plt.rgb.tue_gray, linestyle=':', linewidth=1, alpha=0.8, zorder=1)
    
    # 2b. BASIS-LINIE (0%)
    ax.axhline(0, color=plt.rgb.tue_dark, alpha=0.5, linewidth=1, zorder=1)

    # 3. LINIEN PLOTTEN
    categories = ['GESAMT', 'Sprint', 'Run', 'Jump', 'Throw']
    
    for cat in categories:
        cat_data = final_stats[final_stats['category'] == cat].sort_values('jahr')
        if cat_data.empty: continue
        
        color = colors.get(cat, 'gray')
        # GESAMT bekommt gestrichelte Linie zur Unterscheidung, Rest durchgezogen
        linestyle = '--' if cat == 'GESAMT' else '-'
        
        # Englisches Label nutzen
        label = label_map.get(cat, cat)
        
        ax.plot(cat_data['jahr'], cat_data['rel_change_pct'], 
                label=label,
                color=color,
                linestyle=linestyle,
                marker='.',       
                markersize=4,     
                zorder=3)

    # Styling & Labels (Englisch)
    ax.xaxis.set_major_locator(mtick.MultipleLocator(1))
    
    ax.set_ylabel('Relative Performance Change (\\%)')
    ax.set_xlabel('Year')
    ax.grid(True, alpha=0.3)

        # 1. PANDEMIE-BEREICH (Englisch)
    ax.axvspan(2019.95, 2021.4, color='red', alpha=0.1, label='Pandemic Effect', zorder=0)

    # Legende
    ax.legend(ncols=2, loc='upper left', framealpha=1)

    # plt.ylim(-5, 5)

    if baseline:
        y_lims = ax.get_ylim()
        ax.text(2015.1, y_lims[1]*-0.4, "Baseline", color=plt.rgb.tue_gray, va='top', ha='left')

    if baseline:
        plt.savefig(f'covid_relative_performance_change_{column}_baseline', category="Corona_Analyse")
    else:
        plt.savefig(f'covid_relative_performance_change_{column}', category="Corona_Analyse")
    plt.show()

RuntimeError: Failed to process string with tex because latex could not be found

#### Statistical tests

All years against 2020 (Median MWU and Bootstrap)

In [19]:
import numpy as np
import pandas as pd
from scipy.stats import bootstrap, mannwhitneyu

df_recent = df_stable.copy() # Use the stable cohort for consistency

# 1. Function for the effect size: Relative change of the median
def calculate_relative_diff(sample_2020, sample_baseline):
    m_2020 = np.median(sample_2020)
    m_base = np.median(sample_baseline)
    if m_base == 0: return 0
    return (m_2020 / m_base) - 1

bootstrap_effects = []
categories_to_test = sorted(df_recent['group'].unique().tolist()) + ['TOTAL (All Events)']

for cat in categories_to_test:
    if 'TOTAL' in cat:
        d_2020 = df_recent[df_recent['jahr'] == 2020]['iaaf_score'].values
        d_base = df_recent[df_recent['jahr'] != 2020]['iaaf_score'].values
    else:
        d_2020 = df_recent[(df_recent['group'] == cat) & (df_recent['jahr'] == 2020)]['iaaf_score'].values
        d_base = df_recent[(df_recent['group'] == cat) & (df_recent['jahr'] != 2020)]['iaaf_score'].values

    if len(d_2020) > 15 and len(d_base) > 15:
        # --- BOOTSTRAP ---
        res = bootstrap((d_2020, d_base), calculate_relative_diff, 
                        n_resamples=5000, 
                        confidence_level=0.95, 
                        method='percentile')
        
        ci = res.confidence_interval
        observed_effect = calculate_relative_diff(d_2020, d_base) * 100
        ci_low, ci_high = ci.low * 100, ci.high * 100
        
        # --- MANN-WHITNEY U TEST ---
        # Testing if the performance level in 2020 differs from other years
        u_stat, p_val = mannwhitneyu(d_2020, d_base, alternative='two-sided')
        
        # Significance: Bootstrap CI doesn't contain 0 AND p-value < 0.05
        is_boot_sig = not (ci_low <= 0 <= ci_high)
        is_mwu_sig = p_val < 0.05
        
        bootstrap_effects.append({
            'Category': cat,
            'Effect Size (%)': round(observed_effect, 2),
            '95% CI Low (%)': round(ci_low, 2),
            '95% CI High (%)': round(ci_high, 2),
            'P-Value (MWU)': round(p_val, 4),
            'Sig. (MWU)': "✅ YES" if is_mwu_sig else "❌ NO",
            'Performance Trend': "Increase" if observed_effect > 0 else "Decrease"
        })

# Results Table
df_effects = pd.DataFrame(bootstrap_effects)
display(df_effects.sort_values('Effect Size (%)', ascending=False))

,Category,Effect Size (%),95% CI Low (%),95% CI High (%),P-Value (MWU),Sig. (MWU),Performance Trend
3,Throw,-1.80,-2.51,-0.60,0.0001,✅ YES,Decrease
0,Jump,-1.82,-2.78,-1.02,0.0000,✅ YES,Decrease
2,Sprint,-2.15,-2.65,-1.53,0.0000,✅ YES,Decrease
1,Run,-2.21,-3.05,-1.32,0.0000,✅ YES,Decrease
4,TOTAL (All Events),-2.24,-2.89,-1.82,0.0000,✅ YES,Decrease


2015-19 against 2020 (Median and MWU)

In [20]:
from scipy.stats import mannwhitneyu
import numpy as np
import pandas as pd

# 1. Daten in Blöcke unterteilen
df_baseline = df_recent[(df_recent['jahr'] >= 2015) & (df_recent['jahr'] <= 2019)]
df_2020 = df_recent[df_recent['jahr'] == 2020]

comparison_results = []
categories = sorted(df_recent['group'].unique().tolist()) + ['GESAMT']

for cat in categories:
    # Daten für die jeweilige Kategorie extrahieren
    if cat == 'GESAMT':
        data_base = df_baseline['iaaf_score'].values
        data_2020 = df_2020['iaaf_score'].values
    else:
        data_base = df_baseline[df_baseline['group'] == cat]['iaaf_score'].values
        data_2020 = df_2020[df_2020['group'] == cat]['iaaf_score'].values
    
    if len(data_2020) > 0 and len(data_base) > 0:
        # MWU-Test
        stat, p_val = mannwhitneyu(data_2020, data_base, alternative='two-sided')
        
        # Mediane für die Interpretation
        med_base = np.median(data_base)
        med_2020 = np.median(data_2020)
        rel_diff = (med_2020 / med_base - 1) * 100
        
        comparison_results.append({
            'Kategorie': cat,
            'Median 2015-19': round(med_base, 1),
            'Median 2020': round(med_2020, 1),
            'Diff (%)': round(rel_diff, 2),
            'p-Value': round(p_val, 4),
            'Signifikant': "✅ JA" if p_val < 0.05 else "❌ Nein"
        })

df_final_test = pd.DataFrame(comparison_results)
display(df_final_test.sort_values('Diff (%)'))

,Kategorie,Median 2015-19,Median 2020,Diff (%),p-Value,Signifikant
3,Throw,844.0,820.0,-2.84,0.0000,✅ JA
0,Jump,939.0,916.0,-2.45,0.0000,✅ JA
4,GESAMT,935.0,915.0,-2.14,0.0000,✅ JA
2,Sprint,977.0,958.0,-1.94,0.0000,✅ JA
1,Run,939.0,931.0,-0.85,0.0016,✅ JA


2015-19 against 2020 (IQR and Bootstrap)

In [21]:
import numpy as np
import pandas as pd
from scipy.stats import bootstrap, iqr

# 1. Daten vorbereiten
df_base = df_recent[(df_recent['jahr'] >= 2015) & (df_recent['jahr'] <= 2019)]
df_2020 = df_recent[df_recent['jahr'] == 2020]

# 2. Funktion für die Effektgröße (Relative Änderung des IQR)
def iqr_diff_ratio(sample_2020, sample_baseline):
    iqr_2020 = iqr(sample_2020)
    iqr_base = iqr(sample_baseline)
    if iqr_base == 0: return 0
    return (iqr_2020 / iqr_base) - 1

results_boot = []
categories = sorted(df_recent['group'].unique().tolist()) + ['GESAMT']

print("Running Bootstrap (5000 resamples)...")

for cat in categories:
    if cat == 'GESAMT':
        d_base = df_base['iaaf_score'].values
        d_2020 = df_2020['iaaf_score'].values
    else:
        d_base = df_base[df_base['group'] == cat]['iaaf_score'].values
        d_2020 = df_2020[df_2020['group'] == cat]['iaaf_score'].values

    if len(d_2020) > 15 and len(d_base) > 15:
        # Bootstrap berechnet das KI für die Differenz der IQRs
        res = bootstrap((d_2020, d_base), iqr_diff_ratio, 
                        n_resamples=5000, 
                        confidence_level=0.95, 
                        method='percentile')
        
        ci = res.confidence_interval
        observed = iqr_diff_ratio(d_2020, d_base) * 100
        
        # Signifikanz: Enthält das KI die Null?
        is_sig = not (ci.low <= 0 <= ci.high)
        
        results_boot.append({
            'Kategorie': cat,
            'IQR Änderung (%)': round(observed, 2),
            '95% KI unten': round(ci.low * 100, 2),
            '95% KI oben': round(ci.high * 100, 2),
            'Signifikant': "✅ JA" if is_sig else "❌ Nein"
        })

df_boot_iqr = pd.DataFrame(results_boot)
display(df_boot_iqr.sort_values('IQR Änderung (%)', ascending=False))

Running Bootstrap (5000 resamples)...


,Kategorie,IQR Änderung (%),95% KI unten,95% KI oben,Signifikant
2,Sprint,12.04,4.59,20.75,✅ JA
4,GESAMT,5.33,1.32,9.46,✅ JA
0,Jump,3.73,-7.58,9.78,❌ Nein
3,Throw,1.06,-7.41,10.52,❌ Nein
1,Run,0.73,-7.09,13.97,❌ Nein


2015-19 against 2021-25 (Median and MWU)

In [22]:
from scipy.stats import mannwhitneyu
import numpy as np
import pandas as pd

# 1. Daten in Blöcke unterteilen
df_baseline = df_recent[(df_recent['jahr'] >= 2015) & (df_recent['jahr'] <= 2019)]
df_post = df_recent[(df_recent['jahr'] >= 2021) & (df_recent['jahr'] <= 2025)]

mwu_results = []
categories = sorted(df_recent['group'].unique().tolist()) + ['GESAMT']

print("Berechne Mann-Whitney-U: Baseline (15-19) vs. Post-Corona (21-25)...")

for cat in categories:
    if cat == 'GESAMT':
        d_base = df_baseline['iaaf_score'].values
        d_post = df_post['iaaf_score'].values
    else:
        d_base = df_baseline[df_baseline['group'] == cat]['iaaf_score'].values
        d_post = df_post[df_post['group'] == cat]['iaaf_score'].values

    if len(d_post) > 5 and len(d_base) > 5:
        # MWU-Test (Lageunterschied)
        stat, p_val = mannwhitneyu(d_post, d_base, alternative='two-sided')
        
        # Effekt berechnen (Median-Differenz)
        med_base = np.median(d_base)
        med_post = np.median(d_post)
        rel_diff = (med_post / med_base - 1) * 100
        
        mwu_results.append({
            'Kategorie': cat,
            'N (Base)': len(d_base),
            'N (Post)': len(d_post),
            'Median Diff (%)': round(rel_diff, 2),
            'p-Value': round(p_val, 4),
            'Signifikant': "✅ JA" if p_val < 0.05 else "❌ Nein",
            'Trend': "Besser" if rel_diff > 0 else "Schlechter"
        })

df_mwu_post = pd.DataFrame(mwu_results)
display(df_mwu_post.sort_values('Median Diff (%)', ascending=False))

Berechne Mann-Whitney-U: Baseline (15-19) vs. Post-Corona (21-25)...


,Kategorie,N (Base),N (Post),Median Diff (%),p-Value,Signifikant,Trend
1,Run,4025,4025,2.66,0.0000,✅ JA,Besser
2,Sprint,5075,5075,0.41,0.0046,✅ JA,Besser
4,GESAMT,17500,17500,0.11,0.9359,❌ Nein,Besser
0,Jump,4200,4200,-1.28,0.0000,✅ JA,Schlechter
3,Throw,4200,4200,-2.61,0.0000,✅ JA,Schlechter


2015-19 aigainst 2021-25 IQR and Bootstrap

In [23]:
# 1. Daten vorbereiten: Baseline vs. Erholungsphase
df_base = df_recent[(df_recent['jahr'] >= 2015) & (df_recent['jahr'] <= 2019)]
df_post = df_recent[(df_recent['jahr'] >= 2021) & (df_recent['jahr'] <= 2025)]

results_boot = []
categories = sorted(df_recent['group'].unique().tolist()) + ['GESAMT']

print("Running Bootstrap: Baseline (15-19) vs. Post-Corona (21-24)...")

for cat in categories:
    if cat == 'GESAMT':
        d_base = df_base['iaaf_score'].values
        d_post = df_post['iaaf_score'].values
    else:
        d_base = df_base[df_base['group'] == cat]['iaaf_score'].values
        d_post = df_post[df_post['group'] == cat]['iaaf_score'].values

    if len(d_post) > 15 and len(d_base) > 15:
        # Resampling für den Vergleich der Streuung
        res = bootstrap((d_post, d_base), iqr_diff_ratio, 
                        n_resamples=5000, 
                        confidence_level=0.95, 
                        method='percentile')
        
        ci = res.confidence_interval
        observed = iqr_diff_ratio(d_post, d_base) * 100
        is_sig = not (ci.low <= 0 <= ci.high)
        
        results_boot.append({
            'Kategorie': cat,
            'N (Post)': len(d_post),
            'IQR Änderung (%)': round(observed, 2),
            '95% KI unten': round(ci.low * 100, 2),
            '95% KI oben': round(ci.high * 100, 2),
            'Signifikant': "✅ JA" if is_sig else "❌ Nein"
        })

df_boot_post = pd.DataFrame(results_boot)
display(df_boot_post.sort_values('IQR Änderung (%)', ascending=False))

Running Bootstrap: Baseline (15-19) vs. Post-Corona (21-24)...


,Kategorie,N (Post),IQR Änderung (%),95% KI unten,95% KI oben,Signifikant
3,Throw,4200,14.18,7.59,20.44,✅ JA
1,Run,4025,9.49,5.00,14.39,✅ JA
2,Sprint,5075,9.26,3.64,12.92,✅ JA
4,GESAMT,17500,8.67,5.92,12.24,✅ JA
0,Jump,4200,-3.73,-9.80,0.95,❌ Nein
